# Parameter sweep template

Tests **one strategy** across every instrument and as an equal-weight portfolio, for any number of parameter combinations.

**How to use:** edit the *EDIT HERE* cell — swap the strategy class and the `PARAM_GRID` list — then run all cells below it.

In [ ]:
import pandas as pd
from pathlib import Path

from sysstrat import (
    Asset, Capital, FixedRiskSizer, BacktestRunner, PortfolioRunner,
    load_simple_price_csv,
)
from sysstrat.visualization import plot_backtest_results

# Locate the research repo root (contains data/MCFTR.csv)
ROOT = Path.cwd()
while not (ROOT / "data" / "MCFTR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"

INSTRUMENTS = {
    "MCFTR":  "MCFTR.csv",
    "RGBITR": "RGBITR.csv",
    "GLDRUB": "GLDRUB_TOM.csv",
    "CNYRUB": "CNYRUB_TOM.csv",
    "USDRUB": "USDRUB.csv",
}

assets = {
    t: Asset(ticker=t, price_data=load_simple_price_csv(DATA_DIR / f),
             commission_rate=0.0004, slippage_rate=0.001)
    for t, f in INSTRUMENTS.items()
}

start = max(a.price_data.index.min() for a in assets.values())
end = min(a.price_data.index.max() for a in assets.values())
assets = {t: a.slice(start, end) for t, a in assets.items()}
print(f"Common period: {start.date()} -> {end.date()}")

In [ ]:
capital = Capital(initial_capital=100_000)

# Blended vol performed best in the sizer comparison; switch back to
# FixedRiskSizer(risk_target=0.20, max_leverage=1.0) for plain SMA(252).
sizer = FixedRiskSizer(
    risk_target=0.20,
    vol_method="blended",
    short_span=32,
    long_span=2520,
    max_leverage=1.0,
)

## EDIT HERE — strategy and parameter grid

In [ ]:
from sysstrat.strategies import EWMACStrategy   # <- swap the strategy class

def make_strategy(params: dict):
    """Build the strategy from a parameter dict."""
    return EWMACStrategy(**params)                    # <- adjust kwargs if needed

# Each dict is one parameter combination to test.
PARAM_GRID = [
    {"fast_window": 8,  "slow_window": 32},
    {"fast_window": 16, "slow_window": 64},
    {"fast_window": 32, "slow_window": 128},
    {"fast_window": 64, "slow_window": 256},
]

COMBOS = [
    (", ".join(f"{k}={v}" for k, v in params.items()), params)
    for params in PARAM_GRID
]
COMBOS

## Per-instrument results for each parameter set

In [ ]:
sharpe = {}
total_ret = {}
max_dd = {}
for label, params in COMBOS:
    strategy = make_strategy(params)
    sharpe[label] = {}
    total_ret[label] = {}
    max_dd[label] = {}
    for t, a in assets.items():
        m = BacktestRunner(capital, a, sizer).run(strategy).metrics
        sharpe[label][t] = round(m.sharpe_ratio, 2)
        total_ret[label][t] = round(m.total_return_pct, 1)
        max_dd[label][t] = round(m.max_drawdown_pct, 1)

sharpe_df = pd.DataFrame(sharpe).T[list(assets)]
total_ret_df = pd.DataFrame(total_ret).T[list(assets)]
max_dd_df = pd.DataFrame(max_dd).T[list(assets)]
sharpe_df

In [ ]:
total_ret_df

In [ ]:
max_dd_df

## Diversified equal-weight portfolio for each parameter set

In [ ]:
rows = []
portfolio_results = {}
for label, params in COMBOS:
    strategy = make_strategy(params)
    reports = {t: BacktestRunner(capital, a, sizer).run(strategy) for t, a in assets.items()}
    p = PortfolioRunner(capital).run(reports)
    portfolio_results[label] = p.result
    m = p.metrics
    rows.append({
        "params": label,
        "total ret %": round(m.total_return_pct, 1),
        "vol %": round(m.annual_volatility_pct, 2),
        "sharpe": round(m.sharpe_ratio, 2),
        "sortino": round(m.sortino_ratio, 2),
        "max DD %": round(m.max_drawdown_pct, 1),
    })

port_summary = pd.DataFrame(rows).set_index("params")
port_summary

## Portfolio equity curves

In [ ]:
plot_backtest_results(portfolio_results, panels=["equity", "drawdown"])